# Gallstone Data Processing in Jupyter Notebook
This notebook processes the datasets, and splitting subjects into groups for further analysis.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

In [2]:
class GallstoneDataProcessor:
    def __init__(self, data_path):
        """Initialize the processor with the dataset path."""
        self.data_path = data_path
        self.df = pd.read_csv(data_path, low_memory=False)
        self.filter_log = []

    def inspect_data(self):
        """Inspect the dataset: display first rows, info, shape, missing values, and statistics."""
        
        # Step 1: Inspect the first few rows
        print("First few rows of the dataset:")
        print(self.df.head())
        print("\n" + "="*80 + "\n")

        # Step 2: Check datatypes and basic info
        print("Dataset Information:")
        print(self.df.info())
        print("\n" + "="*80 + "\n")

        # Step 3: Check dataset shape
        print(f"Dataset shape: {self.df.shape[0]} rows, {self.df.shape[1]} columns")
        print("\n" + "="*80 + "\n")

        # Step 4: Check for missing values
        print("Missing values per column:")
        print(self.df.isnull().sum())
        print(f"\nTotal missing values: {self.df.isnull().sum().sum()}")
        print("\n" + "="*80 + "\n")

        # Step 5: Statistical summary
        print("Statistical Summary:")
        print(self.df.describe())

    def preprocess_data(self, output_path):
        """ Encode categorical variables and scale numerical features."""
        df_encoded = self.df.copy()

        # Identify categorical columns
        categorical_cols = df_encoded.select_dtypes(include=['object']).columns
        print(f"Categorical columns identified for encoding: {categorical_cols.tolist()}")

        # Create label encoders for each categorical column
        label_encoders = {}
        for col in categorical_cols:
            le = LabelEncoder()
            df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))
            label_encoders[col] = le
            print(f"Encoded column '{col}' with labels: {le.classes_}")
        
        # Normalize/scalar numeric features
        # First column is the target
        feature_cols = df_encoded.columns[1:]
        target_col = df_encoded.columns[0]

        print(f"Feature columns: {list(feature_cols)}")
        print(f"Target column: {target_col}")

        # Separate features and target
        X = df_encoded[feature_cols]
        y = df_encoded[target_col]

        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        # Convert back to DataFrame for better readability
        df_scaled = pd.DataFrame(X_scaled, columns=feature_cols)

        # Combine scaled features with target
        df_final = pd.concat([y.reset_index(drop=True), df_scaled], axis=1)
        self.df = df_final
        print("Data preprocessing completed: categorical encoding and feature scaling applied.")

        # Save the processed data
        self.df.to_csv(output_path, index=False)
        print(f"Preprocessed data saved to {output_path}.")
        

    def stratified_train_test_split(self, output_path):
        """Perform a stratified train-test split based on the 'Gallstone Status' column."""
        
        # Perform stratified split
        train_df, test_df = train_test_split(
            self.df,
            test_size=0.2,
            stratify=self.df['Gallstone Status'],
            random_state=42
        )

        # Store the resulting dataframes 

        # Assign train/test labels to df
        self.df['Set'] = np.where(self.df.index.isin(train_df.index), 'train', 'test')

        # Save to CSV
        self.df.to_csv(output_path, index=False)
        print(f"Stratified train-test split completed. Data saved to {output_path}.")
    

In [3]:
# Initiate and execute
processor = GallstoneDataProcessor('data/dataset-uci.csv')
processor.inspect_data()
processor.preprocess_data('data/gallstone_data_preprocessed.csv')
processor.stratified_train_test_split('data/gallstone_data_train_test_split.csv')

First few rows of the dataset:
   Gallstone Status  Age  Gender  Comorbidity  Coronary Artery Disease (CAD)  \
0                 0   50       0            0                              0   
1                 0   47       0            1                              0   
2                 0   61       0            0                              0   
3                 0   41       0            0                              0   
4                 0   42       0            0                              0   

   Hypothyroidism  Hyperlipidemia  Diabetes Mellitus (DM)  Height  Weight  \
0               0               0                       0     185    92.8   
1               0               0                       0     176    94.5   
2               0               0                       0     171    91.1   
3               0               0                       0     168    67.7   
4               0               0                       0     178    89.6   

   ...  High Density Lipo